# 🇷🇺 Социально-экономические показатели регионов России

Exploratory notebook · Source: [«Если быть точным»](https://tochno.st/datasets/regions_collection) · CC BY 4.0

> **Pre-executed** — outputs are visible on GitHub without running anything.  
> Results below use the **1 000-row stratified sample**. For full 1.97M-row analysis: `make all` then re-run.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT          = Path('..').resolve()
PROCESSED_DIR = ROOT / 'data' / 'processed'
SAMPLES_DIR   = ROOT / 'data' / 'samples'

full = PROCESSED_DIR / 'regions_full.parquet'
if full.exists():
    df = pd.read_parquet(full)
    print(f'Full dataset: {len(df):,} rows')
else:
    df = pd.read_parquet(sorted(SAMPLES_DIR.glob('regions_sample_*.parquet'))[0])
    print(f'Sample: {len(df):,} rows (run make all for full data)')

clean = df[df['value_status'] == 'ok']
print(f'Valid rows: {len(clean):,} ({len(clean)/len(df):.1%})')

Sample: 1,000 rows (run make all for full data)
Valid rows: 929 (92.9%)


## 1. Dataset shape & value_status

In [ ]:
print(df[['section','indicator_code','object_level','year','value_status']].dtypes)
print()
print(df['value_status'].value_counts().to_string())
print()
print('Year range:', df['year'].min(), '—', df['year'].max())

section             str
indicator_code      str
object_level        str
year              int32
value_status        str
dtype: object

value_status
ok         929
hidden      42
no_data     29

Year range: 2001 — 2025


## 2. Indicator Catalogue — 1 294 indicators

In [ ]:
cat = pd.read_parquet(PROCESSED_DIR / 'catalogue.parquet')
print(f'{len(cat)} indicators, {cat["section"].nunique()} sections')
cat.head(10)

1294 indicators, 113 sections


,indicator_code,indicator_name,section,primary_unit
0,Y477010001,Выбывшие из субъектов Российской Федерации в отдельные страны,Миграция,Человек
1,Y477010002,Городское население субъектов Российской Федерации на 1 января,Численность населения,Тысяч человек
2,Y477010003,Население субъектов Российской Федерации на 1 января,Численность населения,Тысяч человек
3,Y477010004,Прерывание беременности (аборты),Рождаемость,NaN
4,Y477010005,Прибывшие в субъекты Российской Федерации из отдельных стран,Миграция,Человек
5,Y477010006,Сельское население субъектов Российской Федерации на 1 января,Численность населения,Тысяч человек
6,Y477010007,Средний возраст населения на 1 января,Численность населения,Лет
7,Y477020001,"Женщины и мужчины, занятые на работах с вредными и (или) опасными условиями труда, в обрабатывающих производствах, на конец года","Занятость, безработица и заработная плата","Без субъектов малого предпринимательства, в процентах от общей численности работников организаций соответствующего вида деятельности"
8,Y477020002,"Женщины и мужчины, занятые на работах с вредными и (или) опасными условиями труда, в организациях информации и связи, на конец года","Занятость, безработица и заработная плата","Без субъектов малого предпринимательства, в процентах от общей численности работников организаций соответствующего вида деятельности"
9,Y477020003,"Женщины и мужчины, занятые на работах с вредными и (или) опасными условиями труда, в организациях по водоснабжению, водоотведению, организации сбора и утилизации отходов, деятельности по ликвидации загрязнений, на конец года","Занятость, безработица и заработная плата","Без субъектов малого предпринимательства, в процентах от общей численности работников организаций соответствующего вида деятельности"


## 3. Search catalogue by keyword

In [ ]:
results = cat[cat['indicator_name'].str.contains('заработная плата', case=False, na=False)]
print(f'Found {len(results)} matches')
pd.set_option('display.max_colwidth', 65)
results[['indicator_code','indicator_name','section','primary_unit']].head(10)

Found 16 matches


,indicator_code,indicator_name,section,primary_unit
18,Y477020012,Средняя начисленная заработная плата женщин и мужчин за октябрь,"Занятость, безработица и заработная плата",NaN
142,Y477040034,Среднемесячная номинальная начисленная заработная плата работников в организациях здравоохранения,Занятость и оплата труда в здравоохранении в российской федерации,Рублей
666,Y477110208,Медианная заработная плата работников организаций,Уровень жизни населения,"Без субъектов малого предпринимательства, по данным выборочных обследований, за апрель, рублей"
820,Y477110362,Реальные денежные доходы: Реальная начисленная заработная плата работников организаций,Уровень жизни населения,В процентах к предыдущему году
833,Y477110375,Среднемесячная начисленная заработная плата муниципальных служащих органов местного самоуправления: В местных администрациях (исполнительно-распорядительных органах муниципальных образований),Уровень жизни населения,Рублей
834,Y477110376,Среднемесячная начисленная заработная плата муниципальных служащих органов местного самоуправления: В представительных органах муниципальных образований,Уровень жизни населения,Рублей
835,Y477110377,Среднемесячная начисленная заработная плата муниципальных служащих органов местного самоуправления: Всего,Уровень жизни населения,Рублей
836,Y477110378,Среднемесячная номинальная начисленная заработная плата работников организаций,Уровень жизни населения,Рублей
1025,Y477130013,Номинальная и реальная начисленная заработная плата работников организаций,Денежные доходы населения и их использование,NaN
1127,Y477140048,Средняя начисленная заработная плата работников строительных организаций по категориям персонала,Строительные организации,"Без субъектов малого предпринимательства, по данным выборочного обследования организаций за октябрь, рублей"


## 4. Gross Regional Product (sample rows)

In [ ]:
grp = clean[
    clean['indicator_name'].str.contains('Валовой региональный продукт', na=False) &
    (clean['object_level'] == 'Регион')
]
print(f'{len(grp)} GRP rows in sample')
grp[['object_name','year','indicator_value','indicator_unit']].sort_values('indicator_value',ascending=False).head(10)

7 GRP rows in sample


,object_name,year,indicator_value,indicator_unit
416,Краснодарский край,2024,13507.038626,"В хозяйствах всех категорий, тысяч тонн; для значений в целом по России: млн т"
940,Ставропольский край,2001,456.675000,"В хозяйствах всех категорий, тысяч тонн; для значений в целом по России: млн т"
32,Ставропольский край,2010,324.400000,"В хозяйствах всех категорий, тысяч тонн"
27,Республика Адыгея,2021,106.865780,"В хозяйствах всех категорий, тысяч тонн"
232,Астраханская область,2022,16.661816,"В хозяйствах всех категорий, тысяч тонн"
149,Московская область,2002,0.118000,"В хозяйствах всех категорий, тысяч тонн"
480,Ханты-Мансийский автономный округ — Югра,2019,0.000900,"В хозяйствах всех категорий, тысяч тонн"


## 5. Wage rows in sample

In [ ]:
wages = clean[
    clean['indicator_name'].str.contains('Среднемесячная', na=False) &
    clean['indicator_name'].str.contains('заработная плата', na=False) &
    (clean['object_level'] == 'Федеральный округ')
]
print(f'{len(wages)} wage rows for federal districts in sample')
wages[['object_name','year','indicator_value','indicator_unit']].sort_values(['object_name','year']).head(12)

1 wage rows for federal districts in sample


,object_name,year,indicator_value,indicator_unit
599,Северо-Западный федеральный округ,2019,45745.0,Рублей


## 6. Section Coverage — all 117 sections

In [ ]:
secs = pd.read_parquet(PROCESSED_DIR / 'sections.parquet')
pd.set_option('display.max_colwidth', 55)
pd.set_option('display.max_rows', 120)
secs

,section,row_count,indicator_count,year_min,year_max,pct_ok
0,"Сельское, лесное хозяйство, рыболовство и рыбоводство",150528,72,2001,2024,0.8034
1,Уровень жизни населения,132336,47,2001,2025,0.9748
2,Население,79273,38,2001,2024,0.9668
3,Наука и инновации,75744,33,2001,2024,0.9465
4,Финансы,73808,34,2001,2025,0.9430
5,Субъекты малого и среднего предпринимательства — юридические лица,69984,77,2010,2023,0.6327
6,Здравоохранение,66912,32,2001,2024,0.9883
7,Образование,56544,28,2001,2024,0.9564
8,Цены и тарифы,52752,25,2001,2024,0.9635
9,Состояние здоровья населения,52128,22,2010,2024,0.9913


## 7. Territory Reference — 98 territories

In [ ]:
objects = pd.read_parquet(PROCESSED_DIR / 'objects.parquet')
print(objects['object_level'].value_counts().to_string())
objects.head(10)

object_level
Регион               89
Федеральный округ     8
Страна                1


,object_name,object_level,object_oktmo,object_okato
0,Алтайский край,Регион,01000000,01000000
1,Амурская область,Регион,10000000,10000000
2,Архангельская область (без автономного округа),Регион,11000000,11000000
3,Архангельская область (с автономным округом),Регион,11000000,11000000
4,Архангельская область (с автономным округом),Регион,11700000,11200000
5,Астраханская область,Регион,12000000,12000000
6,Белгородская область,Регион,14000000,14000000
7,Брянская область,Регион,15000000,15000000
8,Владимирская область,Регион,17000000,17000000
9,Волгоградская область,Регион,18000000,18000000


---
*Росстат / «Если быть точным», 2026. CC BY 4.0 · [tochno.st/datasets/regions_collection](https://tochno.st/datasets/regions_collection)*